# 开放数据集上的质检、覆盖和四级标注

还没有自有头戴设备时，用带手部位姿的开放数据把规格 v10 的前半跑通：**统一 episode → QC/产出率 → 覆盖空档**，校验四级标注，再把 QC 通过的片段导出成 LeRobot v3.0，用线性 BC 打几步并把 `val_loss` 交给缩放律。差距分析见 `docs/gap_analysis.md`，定义见 `docs/headcam_data_spec.md` §8。

默认只跑**合成的 EgoDex 式 HDF5**（两条短片段），不下载数据。真实 EgoDex 测试集 `test.zip` 约 16.1GB，许可 CC BY-NC-ND，不要提交进仓库。

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/skyxiaobai/robot/blob/main/notebooks/open_dataset_pipeline_colab.ipynb)


In [ ]:
%pip install -q numpy h5py pyarrow pandas matplotlib


In [ ]:
import os
import subprocess
from pathlib import Path

root = Path.cwd()
if not (root / "scripts" / "demo_open_dataset_pipeline.py").exists():
    subprocess.check_call([
        "git", "clone", "--depth", "1",
        "https://github.com/skyxiaobai/robot.git", "/content/robot",
    ])
    os.chdir("/content/robot")
print("repo", Path.cwd())


## 合成样本

一条手腕在画面中央移动，一条置信度很低且手腕在画面外。产出率应约为 50%：只有通过片段级 QC 的帧计入可用数据。两条都缺 SUBTASK 和 INSTRUCTION，因为源数据只有整段描述。


In [ ]:
import subprocess
import sys
from pathlib import Path

out = Path("outputs/open_data_demo")
subprocess.check_call([
    sys.executable, "scripts/demo_open_dataset_pipeline.py",
    "--out", str(out),
])
print((out / "yield_episodes.csv").read_text(encoding="utf-8"))
print((out / "coverage_counts.csv").read_text(encoding="utf-8"))


In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, "scripts/validate_hierarchy.py",
    "examples/hierarchy_annotation.json", "--strict",
])


## 换成 EgoDex 测试集抽样

只在磁盘够的时候做。下载和解压都不在默认单元里。

```bash
curl -L "https://ml-site.cdn-apple.com/datasets/egodex/test.zip" -o test.zip
unzip test.zip
python scripts/demo_open_dataset_pipeline.py --input test --out outputs/egodex_sample --limit 20
```

`--limit 20` 只转换前 20 条 HDF5。转换结果里的世界系是 ARKit 原点，不是本仓库重新做的 SLAM。严格校验会失败，直到有人补上时间分段子任务和分手指令。pusht 上的 ACT 仍走 `notebooks/act_pusht_colab.ipynb`。同一份产出率 CSV 可以接下面的 LeRobot 导出：把 `--episodes` 和 `--yield-csv` 换成 `outputs/egodex_sample/unified` 与 `outputs/egodex_sample/yield_episodes.csv`。


## 导出 LeRobot 并做几步线性 BC

接上面合成演示的 `outputs/open_data_demo`。只导出产出率 CSV 里 `accepted=yes` 的片段。合成样本没有 mp4，导出会写 16×16 占位视频（需要系统里的 ffmpeg，Colab 默认有）。

`observation.state` 是双手关节加当前手腕（140 维），`action` 是**下一帧**双手手腕（14 维），片段最后一帧丢掉。语言优先用该帧的 SUBTASK，没有就用 TASK.instruction。策略是 CPU 上的线性行为克隆，不安装 torch，也不改 pusht 上的 ACT。`--max-frames` 截断帧数，给缩放律两个不同数据量。

In [ ]:
import subprocess
import sys
from pathlib import Path

out = Path("outputs/open_data_demo")
dataset = Path("outputs/egodex_lerobot")
subprocess.check_call([
    sys.executable, "scripts/ego_to_lerobot.py",
    "--episodes", str(out / "unified"),
    "--yield-csv", str(out / "yield_episodes.csv"),
    "--out", str(dataset),
])
log_all = Path("outputs/ego_pretrain_all.log")
log_small = Path("outputs/ego_pretrain_8.log")
subprocess.check_call([
    sys.executable, "scripts/ego_pretrain_bc.py",
    "--dataset", str(dataset), "--steps", "5",
    "--log", str(log_all),
])
subprocess.check_call([
    sys.executable, "scripts/ego_pretrain_bc.py",
    "--dataset", str(dataset), "--steps", "5",
    "--max-frames", "8",
    "--log", str(log_small),
])
print(log_small.read_text(encoding="utf-8"))
print(log_all.read_text(encoding="utf-8"))

In [ ]:
import subprocess
import sys
from pathlib import Path

def _frames(text):
    for line in text.splitlines():
        if line.startswith("policy="):
            for part in line.split():
                if part.startswith("frames="):
                    return part.split("=", 1)[1]
    raise SystemExit("日志里没有 frames=")

log_all = Path("outputs/ego_pretrain_all.log")
log_small = Path("outputs/ego_pretrain_8.log")
runs = Path("outputs/ego_pretrain_runs.yaml")
runs.write_text(
    "runs:\n"
    "  - name: ego_small\n"
    "    size: %s\n"
    "    log: ego_pretrain_8.log\n"
    "  - name: ego_all\n"
    "    size: %s\n"
    "    log: ego_pretrain_all.log\n" % (
        _frames(log_small.read_text(encoding="utf-8")),
        _frames(log_all.read_text(encoding="utf-8")),
    ),
    encoding="utf-8",
)
html = Path("outputs/ego_scaling.html")
subprocess.check_call([
    sys.executable, "scripts/scaling_law.py",
    "--runs", str(runs),
    "--out", str(html),
])
print(html.read_text(encoding="utf-8")[:400])